# Private Tailscale + Rustygate Tunnel & Deploy (`tunnel`)

> Self-healing private Tailscale + `rustygate` tunnel setup for `clikernel`, plus single-zip `dist/` deployment to a Solveit test instance.

Neither your local Mac nor the Solveit instance is exposed to the public internet: `rustygate` binds strictly to `127.0.0.1:8787` with a 32-byte bearer token and `--cull-idle 1800`, and `tailscale serve` exposes it only inside your private Tailscale tailnet (never `tailscale funnel`).

In [ ]:
#| default_exp tunnel

In [ ]:
#| export
import asyncio, base64, io, json, secrets, subprocess, tomllib, zipfile
from urllib.request import Request, urlopen
from fastcore.utils import *

from craft_dev.auth import DEFAULT_AUTH_FILE, REPO_ROOT, _write_private, default_target_url
from craft_dev.build import CRAFT_ASSET_FILES, CRAFT_ROOT_ASSETS, DEFAULT_DIST_DIR
from craft_dev.cdp import open_solveit

In [ ]:
#| export
DEFAULT_RG_TOKEN_FILE = REPO_ROOT/'.auth'/'craft-test-rustygate.token'
DEFAULT_TS_TAG = 'tag:solveit-sandbox'
GATEWAYS_TOML = Path.home()/'.config'/'clikernel'/'gateways.toml'

REMOTE_TUNNEL_SCRIPT = r"""#!/usr/bin/env bash
set -euo pipefail
BIN_DIR="/app/data/.local/bin"
STATE_DIR="/app/data/.tailscale"
SOCK="/tmp/tailscaled.sock"
TOK_FILE="/app/data/.rustygate-token"
HOSTNAME_ARG="${1:-craft-test}"
TAG_ARG="${2:-}"
mkdir -p "$BIN_DIR" "$STATE_DIR"
export PATH="$BIN_DIR:$PATH"

if [[ ! -x "$BIN_DIR/tailscale" || ! -x "$BIN_DIR/tailscaled" ]]; then
  TGZ_NAME=$(curl -fsSL https://pkgs.tailscale.com/stable/ | grep -oE 'tailscale_[0-9.]+_amd64\.tgz' | head -n1)
  curl -fsSL "https://pkgs.tailscale.com/stable/${TGZ_NAME}" -o /tmp/tailscale.tgz
  tar -xzf /tmp/tailscale.tgz -C /tmp
  DIR="/tmp/${TGZ_NAME%.tgz}"
  cp "$DIR/tailscale" "$DIR/tailscaled" "$BIN_DIR/"
  chmod 755 "$BIN_DIR/tailscale" "$BIN_DIR/tailscaled"
  rm -rf /tmp/tailscale.tgz "$DIR"
fi

if ! pgrep -f "tailscaled.*$SOCK" >/dev/null 2>&1; then
  nohup "$BIN_DIR/tailscaled" --tun=userspace-networking --statedir="$STATE_DIR" --socket="$SOCK" >/tmp/tailscaled.log 2>&1 </dev/null &
  for _ in $(seq 1 30); do [[ -S "$SOCK" ]] && break; sleep 0.2; done
fi

STATE=$("$BIN_DIR/tailscale" --socket="$SOCK" status --json 2>/dev/null | python3 -c "import sys,json; print(json.load(sys.stdin).get('BackendState',''))" 2>/dev/null || true)
TAG_FLAGS=()
if [[ -n "$TAG_ARG" ]]; then TAG_FLAGS=(--advertise-tags="$TAG_ARG"); fi
if [[ "$STATE" != "Running" ]]; then
  OUT=$("$BIN_DIR/tailscale" --socket="$SOCK" up --hostname="$HOSTNAME_ARG" --accept-dns=false "${TAG_FLAGS[@]}" --timeout=15s 2>&1 || true)
  URL=$(printf '%s\n' "$OUT" | grep -oE 'https://login\.tailscale\.com/[^[:space:]]+' | head -n1 || true)
  if [[ -n "$URL" ]]; then
    echo "TUNNEL_AUTH_URL url=$URL"
    exit 0
  fi
fi

if ! pgrep -f "rustygate.*--port 8787" >/dev/null 2>&1; then
  nohup /usr/local/bin/rustygate --host 127.0.0.1 --port 8787 --root /app/data --token-file "$TOK_FILE" --cull-idle 1800 >/tmp/rustygate.log 2>&1 </dev/null &
  sleep 1
fi

"$BIN_DIR/tailscale" --socket="$SOCK" funnel reset >/dev/null 2>&1 || true
"$BIN_DIR/tailscale" --socket="$SOCK" serve reset >/dev/null 2>&1 || true
MODE="https"
if ! "$BIN_DIR/tailscale" --socket="$SOCK" serve --bg --https=8787 http://127.0.0.1:8787 >/tmp/ts-serve.log 2>&1; then
  MODE="tcp"
  "$BIN_DIR/tailscale" --socket="$SOCK" serve --bg --tcp 8787 tcp://127.0.0.1:8787 >/tmp/ts-serve.log 2>&1
fi

TS_IP=$("$BIN_DIR/tailscale" --socket="$SOCK" ip -4 | head -n1)
TS_DNS=$("$BIN_DIR/tailscale" --socket="$SOCK" status --json | python3 -c "import sys,json; d=json.load(sys.stdin); print((d.get('Self',{}).get('DNSName') or '').rstrip('.'))")
echo "TUNNEL_OK mode=$MODE ip=$TS_IP dns=$TS_DNS"
"""

In [ ]:
#| export
def _load_or_create_token(path=DEFAULT_RG_TOKEN_FILE):
    p = Path(path)
    if p.is_file() and len(p.read_text().strip()) >= 32:
        tok = p.read_text().strip()
        os.chmod(p, 0o600)
    else:
        tok = secrets.token_hex(32)
        _write_private(p, tok + '\n')
    return tok

def ensure_local_tailscale(down=False):
    "Bring local Tailscale up (or down when `down=True`) headlessly via the CLI."
    cmd = ['tailscale', 'down' if down else 'up']
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode: raise RuntimeError(f"{' '.join(cmd)} failed: {res.stderr.strip() or res.stdout.strip()}")
    return not down

In [ ]:
#| export
def read_gateway_entry(name='craft-test', path=GATEWAYS_TOML):
    "Read `[gateways.<name>]` from `gateways.toml`, or `None` if absent."
    p = Path(path)
    if not p.is_file(): return None
    return tomllib.loads(p.read_text()).get('gateways', {}).get(name)

def write_gateway_entry(name, url, token, path=GATEWAYS_TOML):
    "Upsert `[gateways.<name>]` in `gateways.toml` with `0600` permissions, preserving other sections and comments."
    p = Path(path)
    hdr = f"[gateways.{name}]" if re.fullmatch(r'[A-Za-z0-9_-]+', name) else f"[gateways.{json.dumps(name)}]"
    block = [hdr, f"url = {json.dumps(url)}", f"token = {json.dumps(token)}"]
    lines = p.read_text().splitlines() if p.is_file() else []
    tbl_re = re.compile(r'^\s*\[([^\]]+)\]\s*(?:#.*)?$')
    start = next((i for i, ln in enumerate(lines)
        if (m := tbl_re.match(ln)) and m.group(1).strip() in (f"gateways.{name}", f"gateways.{json.dumps(name)}")), None)
    if start is not None:
        end = next((i for i in range(start + 1, len(lines)) if tbl_re.match(lines[i])), len(lines))
        tail = lines[end:]
        out = lines[:start] + block + ([''] + tail if tail else [])
    else:
        while lines and not lines[-1].strip(): lines.pop()
        out = (lines + [''] if lines else []) + block
    _write_private(p, '\n'.join(out).rstrip() + '\n')
    return p

def tunnel_healthy(url, token, timeout=4.0):
    "Return `True` if `rustygate` responds at `url` with valid bearer auth."
    req = Request(url.rstrip('/') + '/mcp', data=b'{}',
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'})
    try:
        with urlopen(req, timeout=timeout) as r: return r.status < 500
    except Exception as e:
        code = getattr(e, 'code', None)
        return code is not None and code not in (401, 403) and code < 500

In [ ]:
#| export
def _parse_tunnel_ok(text):
    if m_auth := re.search(r'TUNNEL_AUTH_URL\s+url=(\S+)', text or ''): return {'status': 'needs-login', 'auth_url': m_auth.group(1)}
    m = re.search(r'TUNNEL_OK\s+mode=(\S+)\s+ip=(\S+)\s+dns=(\S*)', text or '')
    if not m: raise RuntimeError(f"Remote tunnel bootstrap did not report TUNNEL_OK:\n{text}")
    mode, ip, dns = m.groups()
    url = f"https://{dns}:8787" if mode == 'https' and dns else f"http://{ip}:8787"
    return dict(status='started', mode=mode, ip=ip, dns=dns, url=url)

def ensure_tunnel(
    name='craft-test', # Gateway name in `~/.config/clikernel/gateways.toml` and Tailscale hostname
    *,
    url=None, # Solveit dialog URL on the target instance (defaults to saved config / `SOLVEIT_TEST_URL`)
    auth_file=DEFAULT_AUTH_FILE, # Saved Solveit cookie JSON
    rg_token_file=DEFAULT_RG_TOKEN_FILE, # Local copy of the 32-byte rustygate bearer token (`chmod 600`)
    tag=DEFAULT_TS_TAG, # Optional Tailscale ACL tag (e.g. `tag:solveit-sandbox`) for one-way isolation
    gateways_path=GATEWAYS_TOML, # Path to `clikernel`'s `gateways.toml`
    force=False, # Re-run remote tunnel script even if currently healthy
):
    "Ensure local Tailscale and remote keyless `rustygate` tunnel are up and registered in `gateways.toml`."
    ensure_local_tailscale(down=False)
    existing = read_gateway_entry(name, gateways_path)
    if not force and existing and tunnel_healthy(existing['url'], existing.get('token', '')):
        return dict(status='already-up', name=name, url=existing['url'])

    rg_tok = _load_or_create_token(rg_token_file)
    target_url = url or default_target_url()
    if not target_url: raise SystemExit("Pass --url or save a default Solveit dialog URL in .auth/config.json.")

    setup_py = (
        "import os, subprocess, tempfile\nfrom pathlib import Path\n"
        f"rg_tok = {rg_tok!r}\n"
        "tok = Path('/app/data/.rustygate-token')\n"
        "if not tok.is_file() or tok.read_text().strip() != rg_tok:\n"
        "    with tempfile.NamedTemporaryFile('w', dir=tok.parent, prefix='.rg-tok-', delete=False) as f:\n"
        "        tmp = Path(f.name); f.write(rg_tok + '\\n'); f.flush(); os.chmod(tmp, 0o600)\n"
        "    os.replace(tmp, tok)\n"
        "    subprocess.run(['pkill', '-f', 'rustygate.*--port 8787'], check=False)\n"
        "else:\n"
        "    os.chmod(tok, 0o600)\n"
        f"p = Path('/app/data/.craft-tunnel.sh'); p.write_text({REMOTE_TUNNEL_SCRIPT!r}); os.chmod(p, 0o700)\n"
        f"r = subprocess.run(['bash', str(p), {name!r}, {tag or ''!r}], capture_output=True, text=True, timeout=90)\n"
        "print('RC:', r.returncode); print(r.stdout)\n"
        "if r.stderr: print('ERR:', r.stderr)\n")

    async def _run_bootstrap():
        async with open_solveit(url=target_url, mode='vanilla', auth_file=auth_file, wait_ms=1500) as (page, _, _):
            if not await page.sel_exists('#dialog-container'):
                raise SystemExit(
                    f"Solveit instance {name!r} appears to be powered off (no dialog found at {target_url}).\n"
                    f"Turn on {name!r} at https://solve.it.com/dashboard and re-run `craft-cdp tunnel`.")
            before_ids = await page.eval(
                "[...document.querySelectorAll('#dialog-container > .editable[id]')].map(el => el.id)")
            code_js, before_js = json.dumps(setup_py), json.dumps(before_ids)
            await page.eval(f"""(async () => {{
              const kid = document.getElementById('kid').value;
              await window.htmx.ajax('POST', '/upsert_msg_', {{source: '#kid', swap: 'none', values: {{kid, id_: '', msg_type: 'code', is_input: '1', content: {code_js}}}}});
            }})()""")
            await page.wait_for(
                f"(() => {{ const before = new Set({before_js}); return [...document.querySelectorAll('#dialog-container > .editable[id]')].some(el => !before.has(el.id)); }})()",
                timeout=15)
            cell_id = await page.eval(f"""(() => {{
              const before = new Set({before_js});
              const el = [...document.querySelectorAll('#dialog-container > .editable[id]')].find(x => !before.has(x.id));
              return el ? el.id.replace(/^_/, '') : null;
            }})()""")
            if not cell_id: raise RuntimeError("Could not locate temporary tunnel bootstrap cell.")
            cid_js = json.dumps(cell_id)
            try:
                await page.eval(f"""(async () => {{
                  const kid = document.getElementById('kid').value;
                  await window.htmx.ajax('POST', '/add_runq_', {{source: '#kid', swap: 'none', values: {{kid, id_: {cid_js}, cmd: 'run'}}}});
                }})()""")
                await page.wait_for(
                    f"(() => {{ const o = document.querySelector('#_{cell_id} .card-out, #_{cell_id} .msg-output'); return !!(o && (o.innerText.includes('TUNNEL_OK') || o.innerText.includes('TUNNEL_AUTH_URL') || o.innerText.includes('RC:'))); }})()",
                    timeout=95)
                return await page.eval(f"document.querySelector('#_{cell_id} .card-out, #_{cell_id} .msg-output')?.innerText || ''")
            finally:
                await page.eval(f"""(async () => {{
                  const kid = document.getElementById('kid').value;
                  await window.htmx.ajax('POST', '/rm_msg_', {{source: '#kid', swap: 'none', values: {{kid, id_: {cid_js}, ids: {cid_js}}}}});
                }})()""")
    out = asyncio.run(_run_bootstrap())

    info = _parse_tunnel_ok(out)
    if info['status'] == 'needs-login': return {'name': name, **info}
    write_gateway_entry(name, info['url'], rg_tok, gateways_path)
    return {'name': name, **info}

In [ ]:
#| export
def build_deploy_zip(dist_dir=DEFAULT_DIST_DIR):
    "Pack `dist_dir` into an in-memory zip laid out for `/app/data/` (`CRAFT.*` at root, rest under `.craft-assets/`)."
    dist_dir = Path(dist_dir)
    buf, count = io.BytesIO(), 0
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for name in CRAFT_ASSET_FILES:
            p = dist_dir/name
            if not p.is_file(): continue
            arc = name if name in CRAFT_ROOT_ASSETS else f".craft-assets/{name}"
            zf.write(p, arc)
            count += 1
        icons = dist_dir/'vscode-icons'
        if icons.is_dir():
            for p in sorted(icons.rglob('*')):
                if p.is_file():
                    zf.write(p, f".craft-assets/vscode-icons/{p.relative_to(icons)}")
                    count += 1
    return buf.getvalue(), count

In [ ]:
#| export
def _rg_mcp_call(mcp_url, token, method, params=None, sid=None, rid=1, timeout=30):
    hdrs = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json', 'Accept': 'application/json, text/event-stream'}
    if sid: hdrs['Mcp-Session-Id'] = sid
    body = json.dumps(dict(jsonrpc='2.0', id=rid, method=method, params=params or {})).encode()
    with urlopen(Request(mcp_url, data=body, headers=hdrs), timeout=timeout) as r:
        new_sid = r.headers.get('Mcp-Session-Id') or sid
        raw = r.read().decode()
    if raw.startswith('data:'): raw = '\n'.join(ln[5:].strip() for ln in raw.splitlines() if ln.startswith('data:'))
    data = json.loads(raw) if raw.strip() else {}
    if 'error' in data: raise RuntimeError(f"rustygate MCP {method} failed: {data['error']}")
    return new_sid, data.get('result', {})

def deploy_dist(
    dist_dir=DEFAULT_DIST_DIR, # Built CRAFT assets directory
    *,
    name='craft-test', # Gateway name in `~/.config/clikernel/gateways.toml`
    gateways_path=GATEWAYS_TOML, # Path to `clikernel`'s `gateways.toml`
    url=None, # Optional Solveit dialog URL (only used if tunnel needs to be started)
    auth_file=DEFAULT_AUTH_FILE, # Solveit `storage_state` JSON (only used if tunnel needs to be started)
):
    "Deploy `dist_dir` atomically to `/app/data/` on the Solveit instance over the private `rustygate` Tailscale tunnel."
    gw = read_gateway_entry(name, gateways_path)
    if not gw or not tunnel_healthy(gw['url'], gw.get('token', '')):
        res = ensure_tunnel(name=name, url=url, auth_file=auth_file, gateways_path=gateways_path)
        if res['status'] == 'needs-login':
            raise SystemExit(f"Approve Tailscale login first:\n  {res['auth_url']}\nThen re-run `craft-cdp deploy`.")
        gw = read_gateway_entry(name, gateways_path)

    mcp_url, token = gw['url'].rstrip('/') + '/mcp', gw['token']
    zbytes, count = build_deploy_zip(dist_dir)
    b64 = base64.b64encode(zbytes).decode('ascii')
    unpack_code = (
        "import base64, io, os, shutil, tempfile, zipfile\nfrom pathlib import Path\n"
        f"raw = base64.b64decode({b64!r})\n"
        "dst = Path('/app/data').resolve()\n"
        "stage = Path(tempfile.mkdtemp(prefix='.craft-deploy-', dir=dst))\n"
        "try:\n"
        "    staged = []\n"
        "    with zipfile.ZipFile(io.BytesIO(raw)) as zf:\n"
        "        for m in zf.infolist():\n"
        "            if m.is_dir(): continue\n"
        "            rel = Path(m.filename)\n"
        "            if rel.is_absolute() or '..' in rel.parts: raise ValueError(f'Unsafe zip path: {m.filename}')\n"
        "            tmp = (stage / rel).resolve()\n"
        "            if not tmp.is_relative_to(stage): raise ValueError(f'Unsafe zip path: {m.filename}')\n"
        "            tmp.parent.mkdir(parents=True, exist_ok=True)\n"
        "            tmp.write_bytes(zf.read(m))\n"
        "            staged.append((tmp, dst / rel))\n"
        "    for tmp, out in staged:\n"
        "        out.parent.mkdir(parents=True, exist_ok=True)\n"
        "        os.replace(tmp, out)\n"
        "    print('UNPACKED', len(staged))\n"
        "finally:\n"
        "    shutil.rmtree(stage, ignore_errors=True)\n")

    sid, _ = _rg_mcp_call(mcp_url, token, 'initialize', dict(protocolVersion='2025-03-26', capabilities={}, clientInfo={'name': 'craft-deploy', 'version': '0.1'}), rid=1)
    try:
        sid, _ = _rg_mcp_call(mcp_url, token, 'tools/call', {'name': 'create', 'arguments': {'kernel': 'py'}}, sid=sid, rid=2)
        _, res = _rg_mcp_call(mcp_url, token, 'tools/call', {'name': 'exec', 'arguments': {'code': unpack_code}}, sid=sid, rid=3)
        if res.get('isError'): raise RuntimeError(f"Remote unpack failed: {res}")
    finally:
        try: urlopen(Request(mcp_url, method='DELETE', headers={'Authorization': f'Bearer {token}', 'Mcp-Session-Id': sid}), timeout=5)
        except Exception: pass
    return dict(gateway=gw['url'], files=count, bytes=len(zbytes))

Offline tests for `write_gateway_entry`, `_parse_tunnel_ok`, and `build_deploy_zip`.

In [ ]:
import tempfile
from fastcore.test import test_eq

In [ ]:
with tempfile.TemporaryDirectory() as d:
    gw = Path(d)/'gateways.toml'
    gw.write_text('# keep me\n\n[settings]\ntimeout = 30\n')
    write_gateway_entry('craft-test', 'https://craft-test.ts.net:8787', 'tok123', gw)
    write_gateway_entry('other', 'http://127.0.0.1:9999', 'tok456', gw)
    write_gateway_entry('craft-test', 'https://craft-test.ts.net:8787', 'tok789', gw)
    raw = gw.read_text()
    test_eq('# keep me' in raw and '[settings]' in raw, True)
    test_eq(tomllib.loads(raw)['settings']['timeout'], 30)
    test_eq(read_gateway_entry('craft-test', gw), {'url': 'https://craft-test.ts.net:8787', 'token': 'tok789'})
    test_eq(read_gateway_entry('other', gw)['url'], 'http://127.0.0.1:9999')
    test_eq(gw.stat().st_mode & 0o777, 0o600)

test_eq(_parse_tunnel_ok('TUNNEL_OK mode=https ip=100.1.2.3 dns=craft-test.ts.net')['url'], 'https://craft-test.ts.net:8787')
test_eq(_parse_tunnel_ok('TUNNEL_OK mode=tcp ip=100.1.2.3 dns=')['url'], 'http://100.1.2.3:8787')